#Analisis exploratorio de los datos de satelites Starlink extraidos de la API de Space-X. Hay una relación entre los parametros de orbita y el tiempo que un satelite permanece en orbita / la salida del orbite de un satelite?

In [ ]:
#import libraries

import pandas as pd
import numpy as np
import os
import requests
import datetime
import matplotlib.pyplot as plt
from matplotlib import ticker

import seaborn as sns
sns.set()


import warnings
warnings.filterwarnings("ignore")

In [ ]:
#save URL for API call and the output from the API

url_starlink_sats = 'https://api.spacexdata.com/v4/starlink'
res_starlink_sats = requests.get(url = url_starlink_sats)
satellites = res_starlink_sats.json()

In [3]:
satellites

[{'spaceTrack': {'CCSDS_OMM_VERS': '2.0',
   'COMMENT': 'GENERATED VIA SPACE-TRACK.ORG API',
   'CREATION_DATE': '2020-10-13T04:16:08',
   'ORIGINATOR': '18 SPCS',
   'OBJECT_NAME': 'STARLINK-30',
   'OBJECT_ID': '2019-029K',
   'CENTER_NAME': 'EARTH',
   'REF_FRAME': 'TEME',
   'TIME_SYSTEM': 'UTC',
   'MEAN_ELEMENT_THEORY': 'SGP4',
   'EPOCH': '2020-10-13T02:56:59.566560',
   'MEAN_MOTION': 16.43170483,
   'ECCENTRICITY': 0.0003711,
   'INCLINATION': 52.9708,
   'RA_OF_ASC_NODE': 332.0356,
   'ARG_OF_PERICENTER': 120.7278,
   'MEAN_ANOMALY': 242.0157,
   'EPHEMERIS_TYPE': 0,
   'CLASSIFICATION_TYPE': 'U',
   'NORAD_CAT_ID': 44244,
   'ELEMENT_SET_NO': 999,
   'REV_AT_EPOCH': 7775,
   'BSTAR': 0.0022139,
   'MEAN_MOTION_DOT': 0.47180237,
   'MEAN_MOTION_DDOT': 1.2426e-05,
   'SEMIMAJOR_AXIS': 6535.519,
   'PERIOD': 87.635,
   'APOAPSIS': 159.809,
   'PERIAPSIS': 154.958,
   'OBJECT_TYPE': 'PAYLOAD',
   'RCS_SIZE': 'LARGE',
   'COUNTRY_CODE': 'US',
   'LAUNCH_DATE': '2019-05-24',
   'S

In [ ]:
#convert the output from the API into a dataframe
df = pd.DataFrame(satellites)
# The SpaceTrack data is nested inside each satellite JSON. Unnesting the spacetrack data

if 'spaceTrack' in df.columns:
    # Extract the nested spaceTrack dictionaries into separate columns
    space_track_df = pd.json_normalize(df['spaceTrack'])
    
    # Drop the original spaceTrack column 
    df = df.drop('spaceTrack', axis=1)
    
    # Combine the two dataframes
    df = pd.concat([df, space_track_df], axis=1)

# Now I have a DataFrame with each satellite as a row and each key from JSON as a column

df

,launch,version,height_km,latitude,longitude,velocity_kms,id,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,...,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,DECAYED,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,5eb87d30ffd86e000604b378,v0.9,NaN,NaN,NaN,NaN,5eed770f096e59000698560d,2.0,GENERATED VIA SPACE-TRACK.ORG API,2020-10-13T04:16:08,...,US,2019-05-24,AFETR,2020-10-13,1,2850561,163365918,0 STARLINK-30,1 44244U 19029K 20287.12291165 .47180237 1...,2 44244 52.9708 332.0356 0003711 120.7278 242...
1,5eb87d30ffd86e000604b378,v0.9,NaN,NaN,NaN,NaN,5eed770f096e59000698560e,2.0,GENERATED VIA SPACE-TRACK.ORG API,2020-09-28T19:26:08,...,US,2019-05-24,AFETR,2020-09-29,1,2837086,162391575,0 STARLINK-74,1 44293U 19029BL 20272.77440637 .11359350 1...,2 44293 52.9971 48.1405 0002076 323.1313 37...
2,5eb87d30ffd86e000604b378,v0.9,NaN,NaN,NaN,NaN,5eed770f096e59000698560f,2.0,GENERATED VIA SPACE-TRACK.ORG API,2020-10-13T17:46:09,...,US,2019-05-24,AFETR,2020-10-13,1,2851295,163381397,0 STARLINK-29,1 44243U 19029J 20287.71504568 .11982673 1...,2 44243 52.9786 228.8138 0008105 330.1078 31...
3,5eb87d30ffd86e000604b378,v0.9,NaN,NaN,NaN,NaN,5eed770f096e590006985610,2.0,GENERATED VIA SPACE-TRACK.ORG API,2021-09-26T14:36:11,...,US,2019-05-24,AFETR,2021-09-26,1,3129087,185524941,0 STARLINK-76,1 44287U 19029BE 21269.58650490 .32671130 1...,2 44287 52.9578 116.7260 0003663 96.3451 319...
4,5eb87d30ffd86e000604b378,v0.9,NaN,NaN,NaN,NaN,5eed770f096e590006985611,2.0,GENERATED VIA SPACE-TRACK.ORG API,2020-09-02T18:57:38,...,US,2019-05-24,AFETR,2020-09-02,1,2813120,160599026,0 STARLINK-23,1 44237U 19029C 20246.24351608 +.23336896 +1...,2 44237 052.9945 186.9776 0004551 271.9057 198...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3521,None,None,355.305350,44.268273,137.300802,7.700996,63655b77358d5951a1c69dd0,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177338,0 STARLINK-5186,1 54177U 22141W 22334.58335648 .00048875 0...,2 54177 53.2167 337.9278 0003831 6.8795 344...
3522,None,None,353.712986,33.722400,153.947639,7.700405,63655b77358d5951a1c69dfa,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219178207,0 STARLINK-5205,1 54202U 22141AX 22334.58335648 -.00044287 0...,2 54202 53.2156 337.8864 0003705 4.5467 3...
3523,None,None,355.632656,44.561516,136.668403,7.700673,63655b77358d5951a1c69e37,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177167,0 STARLINK-5117,1 54172U 22141R 22334.58335648 -.00074020 0...,2 54172 53.2159 337.9243 0004424 3.2523 347...
3524,None,None,354.996912,40.755340,143.873451,7.700481,63655b77358d5951a1c69e39,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177524,0 STARLINK-5162,1 54184U 22141AD 22334.58335648 -.00088576 0...,2 54184 53.2152 337.9079 0004143 1.3558 356...


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3526 entries, 0 to 3525
Data columns (total 48 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   launch               3215 non-null   object 
 1   version              3215 non-null   object 
 2   height_km            3268 non-null   float64
 3   latitude             3268 non-null   float64
 4   longitude            3268 non-null   float64
 5   velocity_kms         3268 non-null   float64
 6   id                   3526 non-null   object 
 7   CCSDS_OMM_VERS       3526 non-null   object 
 8   COMMENT              3526 non-null   object 
 9   CREATION_DATE        3526 non-null   object 
 10  ORIGINATOR           3526 non-null   object 
 11  OBJECT_NAME          3526 non-null   object 
 12  OBJECT_ID            3526 non-null   object 
 13  CENTER_NAME          3526 non-null   object 
 14  REF_FRAME            3526 non-null   object 
 15  TIME_SYSTEM          3526 non-null   o

In [11]:
df.tail()

,launch,version,height_km,latitude,longitude,velocity_kms,id,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,...,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,DECAYED,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
3521,None,None,355.305350,44.268273,137.300802,7.700996,63655b77358d5951a1c69dd0,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177338,0 STARLINK-5186,1 54177U 22141W 22334.58335648 .00048875 0...,2 54177 53.2167 337.9278 0003831 6.8795 344...
3522,None,None,353.712986,33.722400,153.947639,7.700405,63655b77358d5951a1c69dfa,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219178207,0 STARLINK-5205,1 54202U 22141AX 22334.58335648 -.00044287 0...,2 54202 53.2156 337.8864 0003705 4.5467 3...
3523,None,None,355.632656,44.561516,136.668403,7.700673,63655b77358d5951a1c69e37,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177167,0 STARLINK-5117,1 54172U 22141R 22334.58335648 -.00074020 0...,2 54172 53.2159 337.9243 0004424 3.2523 347...
3524,None,None,354.996912,40.755340,143.873451,7.700481,63655b77358d5951a1c69e39,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219177524,0 STARLINK-5162,1 54184U 22141AD 22334.58335648 -.00088576 0...,2 54184 53.2152 337.9079 0004143 1.3558 356...
3525,None,None,355.396192,44.069184,137.731004,7.700827,63655b77358d5951a1c69e41,2.0,GENERATED VIA SPACE-TRACK.ORG API,2022-11-30T17:18:05,...,US,2022-10-28,AFWTR,None,0,3675646,219178033,0 STARLINK-5122,1 54179U 22141Y 22334.58335648 -.00060135 0...,2 54179 53.2161 337.9252 0003828 2.9089 349...


In [18]:
df[['DECAY_DATE','DECAYED','LAUNCH_DATE']]

,DECAY_DATE,DECAYED,LAUNCH_DATE
0,2020-10-13,1,2019-05-24
1,2020-09-29,1,2019-05-24
2,2020-10-13,1,2019-05-24
3,2021-09-26,1,2019-05-24
4,2020-09-02,1,2019-05-24
...,...,...,...
3521,None,0,2022-10-28
3522,None,0,2022-10-28
3523,None,0,2022-10-28
3524,None,0,2022-10-28


In [ ]:
#i want to create a column for days in orbit. First I need to make sure that decay_date and launch_date are actually correct data types
from datetime import datetime